# CYP Inhibition Prediction: Uncertainty & Chemical-Series Generalization

**A reproducible baseline study for the OpenADMET CYP challenge.**

Research questions:
1. How much does model performance change under random versus scaffold-grouped validation?
2. Does ensemble disagreement correlate with prediction error?
3. Does nearest-training-compound similarity help identify unreliable predictions?
4. Can uncertainty-based abstention reduce error, and does unsupervised rank fusion help?

This notebook uses the **public training data only**. It does not load blinded test labels or make claims about leaderboard performance.

**How to run:** In Google Colab, choose **Runtime → Run all**. The first cell installs dependencies. Runtime may take several minutes depending on package installation and CPU availability.

## 1. Install dependencies

This uses RDKit for molecular parsing and Morgan fingerprints, and scikit-learn for the baseline models.

In [ ]:
%pip -q install "rdkit>=2023.9" "scikit-learn>=1.3" pandas numpy matplotlib seaborn huggingface_hub

## 2. Imports and reproducibility

In [ ]:
import os, sys, json, time, warnings, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold

SEED = 2026
N_BITS = 2048
RADIUS = 2
N_ENSEMBLE = 3  # Increase to 5-10 for a more stable final analysis.
N_TREES = 150   # Increase after the first end-to-end run succeeds.
TEST_SIZE = 0.20

np.random.seed(SEED)
OUTDIR = Path("results")
OUTDIR.mkdir(exist_ok=True)

print("Python:", platform.python_version())
print("RDKit:", rdBase.rdkitVersion)
print("pandas:", pd.__version__)
print("scikit-learn:", __import__("sklearn").__version__)

## 3. Load the public OpenADMET training data

In [ ]:
DATA_URL = (
    "https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/"
    "resolve/main/cyp-challenge-TRAIN_inhibition.csv"
)
data_path = Path("/content/cyp-challenge-TRAIN_inhibition.csv") if Path("/content").exists() else Path("cyp-challenge-TRAIN_inhibition.csv")

if not data_path.exists():
    df = pd.read_csv(DATA_URL)
else:
    df = pd.read_csv(data_path)

print("Shape:", df.shape)
print("Columns:")
for c in df.columns:
    print(" -", c)
display(df.head())

## 4. Identify structure, endpoint, and confidence-interval columns

The code below uses common column-name conventions and prints the mapping it finds. **Inspect this output** before interpreting results. If the dataset schema changes, adjust the mapping explicitly rather than silently guessing.

In [ ]:
ISOFORM_CANDIDATES = ["CYP1A2", "CYP2C9", "CYP2D6", "CYP3A4"]

def find_smiles_column(frame):
    for name in ["SMILES", "smiles", "canonical_smiles", "standardized_smiles"]:
        if name in frame.columns:
            return name
    candidates = [c for c in frame.columns if "smiles" in str(c).lower()]
    if len(candidates) == 1:
        return candidates[0]
    raise ValueError(f"Could not identify SMILES column. Available columns: {list(frame.columns)}")

def find_target_column(frame, isoform):
    # Prefer the exact OpenADMET point-estimate field.
    preferred = f"{isoform}_pIC50_direct_inhibition"
    if preferred in frame.columns:
        return preferred
    candidates = [
        col for col in frame.columns
        if isoform.lower() in str(col).lower()
        and "pic50" in str(col).lower()
        and not any(x in str(col).lower() for x in
                    ("lower", "upper", "low", "high", "conf", "std", "_se", "ci", "err"))
    ]
    return candidates[0] if len(candidates) == 1 else None

def find_ci_columns(frame, isoform, target):
    # The public training data uses _conf_low / _conf_high.
    exact_low = f"{target}_conf_low"
    exact_high = f"{target}_conf_high"
    if exact_low in frame.columns and exact_high in frame.columns:
        return exact_low, exact_high
    low_candidates = [
        col for col in frame.columns if isoform.lower() in str(col).lower()
        and any(x in str(col).lower() for x in ("conf_low", "ci_low", "lower"))
    ]
    high_candidates = [
        col for col in frame.columns if isoform.lower() in str(col).lower()
        and any(x in str(col).lower() for x in ("conf_high", "ci_high", "upper"))
    ]
    return (low_candidates[0] if len(low_candidates) == 1 else None,
            high_candidates[0] if len(high_candidates) == 1 else None)

SMILES_COL = find_smiles_column(df)
TARGET_COLS = {iso: find_target_column(df, iso) for iso in ISOFORM_CANDIDATES}
CI_COLS = {
    iso: find_ci_columns(df, iso, target) if target else (None, None)
    for iso, target in TARGET_COLS.items()
}
print("SMILES column:", SMILES_COL)
print("Target columns:", TARGET_COLS)
print("CI columns:", CI_COLS)
assert all(col is not None and col in df.columns for col in TARGET_COLS.values() if col), "Missing target data"
for iso, (lo, hi) in CI_COLS.items():
    if lo is None or hi is None:
        print(f"Warning: confidence bounds not both available for {iso}; interval diagnostics will be skipped.")

missing_targets = [iso for iso, col in TARGET_COLS.items() if col is None]
if missing_targets:
    raise ValueError(
        f"Could not find target columns for {missing_targets}. "
        "Review the printed columns and edit TARGET_COLS explicitly."
    )

## 5. Parse molecules and generate Morgan fingerprints

In [ ]:
df = df.copy()
df["_mol"] = [Chem.MolFromSmiles(str(s)) if pd.notna(s) else None for s in df[SMILES_COL]]
valid_mol = df["_mol"].notna()
print(f"Valid molecules: {valid_mol.sum():,}/{len(df):,} "
      f"({valid_mol.mean():.1%}); invalid/missing SMILES: {(~valid_mol).sum():,}")

# Invalid structures cannot be featurized. Keep a clean index for all subsequent analyses.
work = df.loc[valid_mol].copy().reset_index(drop=True)
mols = work["_mol"].tolist()

fps = [
    AllChem.GetMorganFingerprintAsBitVect(mol, RADIUS, nBits=N_BITS)
    for mol in mols
]
X = np.zeros((len(fps), N_BITS), dtype=np.uint8)
for i, fp in enumerate(fps):
    DataStructs.ConvertToNumpyArray(fp, X[i])

def scaffold_key(mol):
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    if scaffold.GetNumAtoms() == 0:
        # Acyclic compounds are grouped by canonical SMILES to avoid one giant empty-scaffold group.
        return "ACYCLIC_" + Chem.MolToSmiles(mol, canonical=True)
    return Chem.MolToSmiles(scaffold, canonical=True)

groups = np.asarray([scaffold_key(m) for m in mols], dtype=object)
canonical_smiles = np.asarray([Chem.MolToSmiles(mol, canonical=True) for mol in mols], dtype=object)
print("Unique canonical structures:", len(np.unique(canonical_smiles)))
if len(np.unique(canonical_smiles)) < len(canonical_smiles):
    print("Exact duplicate structures detected; splits will keep duplicates together.")
print("Fingerprint matrix:", X.shape)
print("Unique Murcko/acyclic groups:", len(np.unique(groups)))

## 6. Define metrics and uncertainty signals

Ensemble standard deviation is treated as a **ranking signal**, not as a calibrated confidence interval. Nearest-neighbor similarity is calculated using the maximum ECFP4/Tanimoto similarity to the training fold. A local interval-distance metric is reported only when usable confidence-interval bounds are found.

For a prediction inside a reported interval, interval distance is zero; outside the interval, it is the distance to the nearest bound. This is a transparent local diagnostic and **must not be described as the official competition score** unless verified against the organizers' scoring implementation.

In [ ]:
def interval_distance(y_pred, lower, upper):
    y_pred = np.asarray(y_pred, float)
    lower = np.asarray(lower, float)
    upper = np.asarray(upper, float)
    out = np.maximum(lower - y_pred, 0.0) + np.maximum(y_pred - upper, 0.0)
    invalid = ~np.isfinite(y_pred) | ~np.isfinite(lower) | ~np.isfinite(upper) | (lower > upper)
    out[invalid] = np.nan
    return out

def max_train_similarity(query_fps, train_fps):
    vals = []
    for fp in query_fps:
        sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
        vals.append(max(sims) if sims else np.nan)
    return np.asarray(vals, float)

def risk_coverage(y_true, y_pred, uncertainty, coverages=np.linspace(0.1, 1.0, 10)):
    y_true, y_pred, uncertainty = map(lambda z: np.asarray(z, float), (y_true, y_pred, uncertainty))
    valid = np.isfinite(y_true) & np.isfinite(y_pred) & np.isfinite(uncertainty)
    yt, yp, unc = y_true[valid], y_pred[valid], uncertainty[valid]
    if len(yt) == 0:
        return pd.DataFrame(columns=["coverage", "n_retained", "mae"])
    order = np.argsort(unc)
    rows = []
    for cov in coverages:
        n = max(1, int(np.ceil(cov * len(order))))
        ids = order[:n]
        rows.append({"coverage": cov, "n_retained": n,
                     "mae": mean_absolute_error(yt[ids], yp[ids])})
    return pd.DataFrame(rows)

## 7. Run random and scaffold-grouped validation

We fit each CYP endpoint separately. Each validation strategy uses an ensemble of bootstrap-resampled random forests. We keep the model intentionally modest so the notebook can run in a standard Colab CPU runtime.

The same validation split is used for all ensemble members within an endpoint. Missing target values are removed only for that endpoint. Scaffold grouping is applied before fitting, and no validation labels are used to fit the model.

In [ ]:
def make_split(y, split_kind, seed=SEED):
    idx = np.arange(len(y))
    observed = np.isfinite(y)
    idx_obs = idx[observed]
    if split_kind == "random":
        # Group by canonical molecule so duplicate structures never cross folds.
        splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
        tr_local, va_local = next(splitter.split(idx_obs, groups=canonical_smiles[observed]))
        tr, va = idx_obs[tr_local], idx_obs[va_local]
    elif split_kind == "scaffold":
        splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
        tr_local, va_local = next(splitter.split(idx_obs, y[observed], groups=groups[observed]))
        tr, va = idx_obs[tr_local], idx_obs[va_local]
    else:
        raise ValueError(split_kind)
    return np.asarray(tr), np.asarray(va)

def fit_ensemble_predict(X, y, train_idx, valid_idx, seed=SEED):
    rng = np.random.default_rng(seed)
    predictions = []
    n_train = len(train_idx)
    for member in range(N_ENSEMBLE):
        boot = rng.choice(train_idx, size=n_train, replace=True)
        model = RandomForestRegressor(
            n_estimators=N_TREES,
            max_features=0.7,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=seed + member,
        )
        model.fit(X[boot], y[boot])
        predictions.append(model.predict(X[valid_idx]))
    predictions = np.vstack(predictions)
    return predictions.mean(axis=0), predictions.std(axis=0, ddof=1 if N_ENSEMBLE > 1 else 0)

all_metrics = []
all_predictions = []
all_risk = []
start = time.time()

for iso in ISOFORM_CANDIDATES:
    target_col = TARGET_COLS[iso]
    y = pd.to_numeric(work[target_col], errors="coerce").to_numpy(dtype=float)
    print(f"\n=== {iso}: {target_col}; observed={np.isfinite(y).sum()} ===")
    lo_col, hi_col = CI_COLS[iso]
    for split_kind in ["random", "scaffold"]:
        tr_idx, va_idx = make_split(y, split_kind)
        yhat, unc = fit_ensemble_predict(X, y, tr_idx, va_idx, seed=SEED)
        y_val = y[va_idx]
        sim = max_train_similarity([fps[i] for i in va_idx], [fps[i] for i in tr_idx])
        metrics = {
            "isoform": iso,
            "target": target_col,
            "split": split_kind,
            "n_train": len(tr_idx),
            "n_valid": len(va_idx),
            "mae": mean_absolute_error(y_val, yhat),
            "r2": r2_score(y_val, yhat) if len(y_val) > 1 else np.nan,
            "mean_ensemble_std": float(np.nanmean(unc)),
            "mean_max_train_similarity": float(np.nanmean(sim)),
        }
        ci_width = np.full(len(va_idx), np.nan)
        if lo_col and hi_col:
            lo = pd.to_numeric(work.iloc[va_idx][lo_col], errors="coerce").to_numpy(float)
            hi = pd.to_numeric(work.iloc[va_idx][hi_col], errors="coerce").to_numpy(float)
            ci_width = hi - lo
            ci_width[~np.isfinite(ci_width) | (ci_width < 0)] = np.nan
            ide = interval_distance(yhat, lo, hi)
            metrics["mean_interval_distance"] = float(np.nanmean(ide)) if np.isfinite(ide).any() else np.nan
            metrics["n_with_ci"] = int(np.isfinite(ide).sum())
        # Verify that no exact structures (or scaffold groups in the grouped split)
        # appear in both training and validation.
        assert set(canonical_smiles[tr_idx]).isdisjoint(set(canonical_smiles[va_idx]))
        if split_kind == "scaffold":
            assert set(groups[tr_idx]).isdisjoint(set(groups[va_idx]))
        all_metrics.append(metrics)

        pred_frame = pd.DataFrame({
            "isoform": iso, "split": split_kind, "row_index": va_idx,
            "y_true": y_val, "y_pred": yhat, "abs_error": np.abs(y_val-yhat),
            "ensemble_std": unc, "max_train_similarity": sim,
            "similarity_uncertainty": 1.0 - sim,
            "experimental_ci_width": ci_width,
        })
        # Simple unsupervised rank fusion; uses no validation labels.
        # Ranks are fit only to uncertainty features, not prediction errors.
        pred_frame["combined_uncertainty"] = (
            pred_frame["ensemble_std"].rank(pct=True).to_numpy()
            + pred_frame["similarity_uncertainty"].rank(pct=True).to_numpy()
        ) / 2.0
        all_predictions.append(pred_frame)

        for signal in ["ensemble_std", "similarity_uncertainty", "combined_uncertainty"]:
            rc = risk_coverage(y_val, yhat, pred_frame[signal].to_numpy())
            rc["isoform"] = iso
            rc["split"] = split_kind
            rc["uncertainty_signal"] = signal
            all_risk.append(rc)

        print(f"{split_kind:8s} | n={len(va_idx):4d} | MAE={metrics['mae']:.3f} | "
              f"R2={metrics['r2']:.3f} | ensemble SD={metrics['mean_ensemble_std']:.3f} | "
              f"max similarity={metrics['mean_max_train_similarity']:.3f}")

metrics_df = pd.DataFrame(all_metrics)
predictions_df = pd.concat(all_predictions, ignore_index=True)
risk_df = pd.concat(all_risk, ignore_index=True)
print(f"\nElapsed minutes: {(time.time()-start)/60:.1f}")
display(metrics_df)

## 8. Compare random and scaffold-grouped performance

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=metrics_df, x="isoform", y="mae", hue="split")
plt.ylabel("Validation MAE (pIC50 units)")
plt.title("Random versus scaffold-grouped validation")
plt.tight_layout()
plt.savefig(OUTDIR / "mae_random_vs_scaffold.png", dpi=180)
plt.show()

display(metrics_df.pivot(index="isoform", columns="split", values="mae"))

## 9. Does uncertainty track prediction error?

For each validation prediction, compare absolute error with (a) ensemble disagreement and (b) one minus the nearest-training-compound Tanimoto similarity.

We also examine the average-rank fusion of these two signals and the width of available experimental confidence intervals. The latter is an **analysis-only feature**, not a prediction-time feature. These are associations, not proof of calibrated uncertainty. Evaluate them separately for random and scaffold splits.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharey=False)
for row, split_kind in enumerate(["random", "scaffold"]):
    subset = predictions_df[predictions_df["split"] == split_kind]
    sns.scatterplot(data=subset, x="ensemble_std", y="abs_error", hue="isoform",
                    alpha=0.55, s=22, ax=axes[row, 0], legend=(row == 0))
    axes[row, 0].set_title(f"{split_kind.title()} split: ensemble disagreement")
    axes[row, 0].set_xlabel("Ensemble prediction standard deviation")
    axes[row, 0].set_ylabel("Absolute error")

    sns.scatterplot(data=subset, x="max_train_similarity", y="abs_error", hue="isoform",
                    alpha=0.55, s=22, ax=axes[row, 1], legend=(row == 0))
    axes[row, 1].set_title(f"{split_kind.title()} split: nearest-neighbor similarity")
    axes[row, 1].set_xlabel("Maximum ECFP4/Tanimoto similarity to training set")
    axes[row, 1].set_ylabel("Absolute error")
plt.tight_layout()
plt.savefig(OUTDIR / "uncertainty_vs_error.png", dpi=180)
plt.show()

## 10. Risk–coverage analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
for ax, split_kind in zip(axes, ["random", "scaffold"]):
    sub = risk_df[risk_df["split"] == split_kind]
    sns.lineplot(data=sub, x="coverage", y="mae", hue="uncertainty_signal",
                 style="isoform", markers=False, ax=ax)
    ax.set_title(f"{split_kind.title()} split")
    ax.set_xlabel("Fraction retained (lowest uncertainty first)")
    ax.set_ylabel("MAE among retained predictions")
plt.tight_layout()
plt.savefig(OUTDIR / "risk_coverage.png", dpi=180)
plt.show()

## 11. Summarize association between uncertainty and error

Spearman correlation is a simple rank-based diagnostic. A positive correlation means larger uncertainty tends to accompany larger errors. Interpret small differences cautiously, especially with one split per strategy.

In [ ]:
from scipy.stats import spearmanr

corr_rows = []
for (iso, split_kind), sub in predictions_df.groupby(["isoform", "split"]):
    for signal in ["ensemble_std", "similarity_uncertainty", "combined_uncertainty", "experimental_ci_width"]:
        valid = np.isfinite(sub[signal]) & np.isfinite(sub["abs_error"])
        if valid.sum() >= 3:
            rho, p = spearmanr(sub.loc[valid, signal], sub.loc[valid, "abs_error"])
        else:
            rho, p = np.nan, np.nan
        corr_rows.append({
            "isoform": iso, "split": split_kind, "signal": signal,
            "spearman_rho": rho, "p_value_descriptive_only": p, "n": int(valid.sum())
        })
corr_df = pd.DataFrame(corr_rows)
display(corr_df)

## 12. Save results and record run metadata

The CSVs and figures are written to `results/`. Download them from the Colab file browser or commit selected small outputs to your repository after reviewing them. Do not commit large intermediate model objects.

In [ ]:
metrics_df.to_csv(OUTDIR / "validation_metrics.csv", index=False)
predictions_df.to_csv(OUTDIR / "validation_predictions.csv", index=False)
risk_df.to_csv(OUTDIR / "risk_coverage_data.csv", index=False)
corr_df.to_csv(OUTDIR / "uncertainty_error_correlations.csv", index=False)

metadata = {
    "seed": SEED,
    "n_bits": N_BITS,
    "radius": RADIUS,
    "n_ensemble": N_ENSEMBLE,
    "n_trees": N_TREES,
    "test_size": TEST_SIZE,
    "python": platform.python_version(),
    "rdkit": rdBase.rdkitVersion,
    "pandas": pd.__version__,
    "sklearn": __import__("sklearn").__version__,
    "smiles_column": SMILES_COL,
    "target_columns": TARGET_COLS,
    "ci_columns": CI_COLS,
    "n_input_rows": int(len(df)),
    "n_valid_molecules": int(len(work)),
    "data_url": DATA_URL,
    "n_unique_canonical_smiles": int(len(np.unique(canonical_smiles))),
}
(OUTDIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2, default=str))
print("Saved outputs:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p, f"({p.stat().st_size/1024:.1f} KB)")

## 13. Interpretation checklist

Before writing conclusions, answer these questions from the results:

- [ ] Does scaffold validation consistently have higher error than random validation?
- [ ] Does ensemble disagreement correlate with absolute error?
- [ ] Does nearest-neighbor similarity add information beyond ensemble disagreement?
- [ ] Does uncertainty-based filtering reduce MAE at lower coverage?
- [ ] Are the patterns consistent across the four isoforms?
- [ ] Are conclusions robust to changing the random seed and number of ensemble members?
- [ ] Were CI columns correctly mapped? If not, omit interval-based conclusions until corrected.
- [ ] Have you distinguished exploratory local metrics from the official challenge score?

**Do not claim a scientific discovery based on one split.** For a stronger study, repeat grouped splits across several seeds, report variability, and predefine the analysis before looking at the final results.